### Load packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import matplotlib.gridspec as gridspec
from statannotations.Annotator import Annotator
from scipy.stats import sem
import pickle
import mne
import ipympl

%matplotlib widget

In [ ]:
# statistical annotation
def convert_pvalue_to_asterisks(pvalue):
    if pvalue <= 0.0001:
        return "****"
    elif pvalue <= 0.001:
        return "***"
    elif pvalue <= 0.01:
        return "**"
    elif pvalue <= 0.05:
        return "*"
    return "ns"

In [ ]:
def power_per_band(tfrs, freqs, tmin, tmax, times,freq_bands):
    # select the psds within the time range
    time_mask = (times >= tmin) & (times <= tmax)
    psds = np.mean(tfrs[:,:,time_mask],axis=-1)
    # Initialize a dictionary to store power in each band
    band_powers = []

    # Loop through each frequency band
    for band, (fmin, fmax) in freq_bands.items():
        # Find the indices of the frequencies within the current band
        freq_mask = (freqs >= fmin) & (freqs <= fmax)
        
        # Sum the power in this frequency band
        band_power = psds[:, freq_mask].mean(axis=-1)
        band_powers.append(band_power)

    return np.array(band_powers)

In [ ]:
# Define a function to z-score normalize the epochs
def z_score_normalize(data):
    mean_baseline = np.mean(data, axis=1, keepdims=True)
    std_baseline = np.std(data, axis=1, keepdims=True)
    z_scored_data = (data - mean_baseline) / std_baseline
    # return mne.evoked.EvokedArray(z_scored_data, epochs.info,tmin=epochs.tmin)
    return z_scored_data

In [ ]:
def cal_mean_sem(values):
    values = [x for x in values if ~np.isnan(x)]
    # Calculate the mean
    mean_value = np.mean(values)
    # Calculate the SEM
    sem_value = sem(values)

    return mean_value, sem_value

### load data

In [ ]:
# directory
raw_path = '../../data/eeg'
subj_list = ['A03','A04','A05','A07','A08','A10','A12','A13','A14','A15','A16','A17']

In [ ]:
mne.set_log_level('ERROR')

In [ ]:
with open(raw_path+'/psd_binary.pkl', 'rb') as f:
    all_psd_data_1_nonPE_0, all_psd_data_1_nonPE_1, all_psd_data_1_SPE_0, all_psd_data_1_SPE_2,\
                     all_psd_data_1_RPE_1, all_psd_data_1_RPE_2, all_psd_1_df = pickle.load(f)
    
with open(raw_path+'/psd_navigation.pkl', 'rb') as f:
    all_psd_data_2_nonPE_0, all_psd_data_2_nonPE_1, all_psd_data_2_nonPE_3,\
                      all_psd_data_2_SPE_0, all_psd_data_2_SPE_2, all_psd_data_2_SPE_4,\
                     all_psd_data_2_RPE_1, all_psd_data_2_RPE_2, all_psd_data_2_RPE_4,\
                         all_psd_data_2_dualPE_3, all_psd_data_2_dualPE_4, all_psd_2_df = pickle.load(f)

In [ ]:
# extract eeg configurations from data
with open(raw_path+'/A03_tfr_1.pkl', 'rb') as f:
    tfr = pickle.load(f)
    tfr.apply_baseline((None,None),mode='ratio', verbose='error')

ch_list = tfr.info.ch_names
times = tfr.times
freq_list = tfr.freqs

In [ ]:
# color map
cmap1 = ['#ff9eaa','#3aa6b9','#808080','#6aa84f', '#982b1c'] # SPE, RPE, nonPE, dualPE, PE

In [ ]:
band_list = ['δ\n[1-4Hz]','θ\n[4-8Hz]','α\n[8-14Hz]','β\n[14-31Hz]','γ\n[31-49Hz]']
band_list_simple = ['δ','θ','α','β','γ']
band_list_text = ['Delta','Theta','Alpha','Beta','Gamma']

In [ ]:
pe_1_list = ['nonPE','SPE','RPE'] 
pe_2_list = ['nonPE','SPE','RPE','dualPE']

In [ ]:
freq_bands = {
        'delta': [1, 4],
        'theta': [4, 8],
        'alpha': [8, 14],
        'beta': [14, 31],
        'gamma': [31, 49]
    }
freq_names = ['delta', 'theta', 'alpha', 'beta', 'gamma']

# Stats analysis

### Binary choice task

In [ ]:
p_across_chns_1_nonPE_SPE = [] # nonPE against SPE

# extracting time point for each chn
for ch in range(len(ch_list)):
    ch_p_1 = []

    for f in range(len(freq_names)):
        # x_nonPE = [a[ch][t] for a/ in all_data_nonPE]
        group1 = [a[f][ch] for a in all_psd_data_1_nonPE_0]
        group2 = [b[f][ch] for b in all_psd_data_1_SPE_0]

        stat, p = wilcoxon(group1, group2)
        ch_p_1.append(p)

    p_across_chns_1_nonPE_SPE.append(ch_p_1)

p_across_chns_1_nonPE_SPE = np.array(p_across_chns_1_nonPE_SPE)

In [ ]:
p_across_chns_1_nonPE_RPE = [] # nonPE against RPE

# extracting time point for each chn
for ch in range(len(ch_list)):
    ch_p_1 = []

    for f in range(len(freq_names)):
        # x_nonPE = [a[ch][t] for a/ in all_data_nonPE]
        group1 = [a[f][ch] for a in all_psd_data_1_nonPE_1]
        group2 = [b[f][ch] for b in all_psd_data_1_RPE_1]

        stat, p = wilcoxon(group1, group2)
        ch_p_1.append(p)

    p_across_chns_1_nonPE_RPE.append(ch_p_1)

p_across_chns_1_nonPE_RPE = np.array(p_across_chns_1_nonPE_RPE)

In [ ]:
p_across_chns_1_SPE_RPE = [] # SPE against RPE

# extracting time point for each chn
for ch in range(len(ch_list)):
    ch_p_1 = []

    for f in range(len(freq_names)):
        # x_nonPE = [a[ch][t] for a/ in all_data_nonPE]
        group1 = [a[f][ch] for a in all_psd_data_1_SPE_2]
        group2 = [b[f][ch] for b in all_psd_data_1_RPE_2]

        stat, p = wilcoxon(group1, group2)
        ch_p_1.append(p)

    p_across_chns_1_SPE_RPE.append(ch_p_1)

p_across_chns_1_SPE_RPE = np.array(p_across_chns_1_SPE_RPE)

## Navigation task

In [ ]:
p_across_chns_2_SPE_nonPE = [] #dualPE against RPE

# extracting time point for each chn
for ch in range(len(ch_list)):
    ch_p = []
    for f in range(len(freq_names)):
        group_a = [c[f][ch] for c in all_psd_data_2_SPE_0]
        group_b= [c[f][ch] for c in all_psd_data_2_nonPE_0]

        comparisons = (group_a, group_b)

        stat, p = wilcoxon(comparisons[0], comparisons[1])

        ch_p.append(p)

        
    p_across_chns_2_SPE_nonPE.append(ch_p)


p_across_chns_2_SPE_nonPE = np.array(p_across_chns_2_SPE_nonPE)

In [ ]:
p_across_chns_2_RPE_nonPE = [] #dualPE against RPE

# extracting time point for each chn
for ch in range(len(ch_list)):
    ch_p = []
    for f in range(len(freq_names)):
        group_a = [c[f][ch] for c in all_psd_data_2_RPE_1]
        group_b= [c[f][ch] for c in all_psd_data_2_nonPE_1]

        comparisons = (group_a, group_b)

        stat, p = wilcoxon(comparisons[0], comparisons[1])

        ch_p.append(p)

        
    p_across_chns_2_RPE_nonPE.append(ch_p)


p_across_chns_2_RPE_nonPE = np.array(p_across_chns_2_RPE_nonPE)

In [ ]:
p_across_chns_2_SPE_RPE = [] # SPE aginst RPE
# extracting time point for each chn
for ch in range(len(ch_list)):
    ch_p = []
    for f in range(len(freq_names)):
        group_a = [c[f][ch] for c in all_psd_data_2_SPE_2]
        group_b= [c[f][ch] for c in all_psd_data_2_RPE_2]

        comparisons = (group_a, group_b)

        stat, p = wilcoxon(comparisons[0], comparisons[1])

        ch_p.append(p)

        
    p_across_chns_2_SPE_RPE.append(ch_p)

p_across_chns_2_SPE_RPE = np.array(p_across_chns_2_SPE_RPE)

In [ ]:
p_across_chns_2_dualPE_nonPE = [] #dualPE against SPE

# extracting time point for each chn
for ch in range(len(ch_list)):
    ch_p = []
    for f in range(len(freq_names)):
        group_a = [c[f][ch] for c in all_psd_data_2_dualPE_3]
        group_b= [c[f][ch] for c in all_psd_data_2_nonPE_3]

        comparisons = (group_a, group_b)

        stat, p = wilcoxon(comparisons[0], comparisons[1])

        ch_p.append(p)

        
    p_across_chns_2_dualPE_nonPE.append(ch_p)

p_across_chns_2_dualPE_nonPE = np.array(p_across_chns_2_dualPE_nonPE)

In [ ]:
p_across_chns_2_dualPE = [] #dualPE against RPE, SPE
p_across_chns_2_dualPE_value = [] #dualPE against RPE, SPE

# extracting time point for each chn
for ch in range(len(ch_list)):
    ch_p = []
    ch_p_value = []
    ch_p_1 = []

    for f in range(len(freq_names)):
        group2 = [c[f][ch] for c in all_psd_data_2_SPE_4]
        group3 = [c[f][ch] for c in all_psd_data_2_RPE_4]
        group4 = [c[f][ch] for c in all_psd_data_2_dualPE_4]
        
        p_values = []
        comparisons = [(group2, group4), (group3, group4)]

        for i in range(len(comparisons)):
            stat, p = wilcoxon(comparisons[i][0], comparisons[i][1])
            p_values.append(p)

        # Adjust p-values for multiple comparisons using Bonferroni correction
        _, p_values_corrected, _, _ = multipletests(p_values, method='fdr_bh')
        ch_p_1.append(p_values_corrected)


    for n in range(np.shape(ch_p_1)[0]):
        this_p = ch_p_1[n]
        sig = np.array(this_p)<0.05
        if sig.all():
            ch_p.append(1)
        else:
            ch_p.append(0)
    ch_p_value.append(ch_p_1)
    
    p_across_chns_2_dualPE.append(ch_p)
    p_across_chns_2_dualPE_value.append(ch_p_value)

p_across_chns_2_dualPE = np.array(p_across_chns_2_dualPE)

# Visualization

In [ ]:
# [FINAL-final] no topology - 1 against 1: pickedX
# binary choice task

chs_1 = ['E8','E7','E4']
tenten_chs_1 = ['AFz','FC1','FCz']
band_choice_1 = [[1],[1],[0]]

pe_pairs = [['nonPE','SPE'],['nonPE','RPE'],['RPE','SPE']]
com_list = ['s10','r10','s1r1']
p_value_pairs = [p_across_chns_1_nonPE_SPE, p_across_chns_1_nonPE_RPE, \
                 p_across_chns_1_SPE_RPE]

this_df = all_psd_1_df
group_df = this_df.groupby(['pe_type','band','ch','com'])['power'].mean().reset_index()
colors = {'SPE': cmap1[0], 'RPE': cmap1[1], 'nonPE': cmap1[2]}

cm = 1/2.54 
fig = plt.figure(figsize=(5.5*cm, 3.5*cm))
# Create a GridSPEc object with 2 rows and 2 columns, setting width and height ratios
gs = gridspec.GridSpec(1, 3)
# Create the first subplot, which spans the entire first row (bigger subplot)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2:])
axes = [ax1, ax2, ax3]


for n in range(len(pe_pairs)):
    df = this_df[(this_df['ch']==chs_1[n])&(this_df['com']==com_list[n])]
    pes = pe_pairs[n]
    # scatter for each subject
    x_pe = -1
    for pe in pes:
        for b in range(len(band_choice_1[n])):
            pe_data = df[(df['pe_type'] == pe)&(df['band']==band_choice_1[n][b])]
            axes[n].scatter([b+0.2*x_pe]*len(pe_data), pe_data['power'], label=pe,\
                            edgecolors='None', alpha=.2, color='grey', s=10)
        x_pe = 1
    

    # Manually adding lines between the error bars for the same x
    for i in range(len(band_choice_1[n])):
        pos1 = i + 0.2*(-1)
        pos2 = i + 0.2*(1)
        mean1 = df[(df['band'] == band_choice_1[n][i]) & (df['pe_type'] == pes[0])]['power'].mean()
        mean2 = df[(df['band'] == band_choice_1[n][i]) & (df['pe_type'] == pes[1])]['power'].mean()
        axes[n].plot([pos1, pos2], [mean1, mean2], color='black', linestyle='-',linewidth=.8)

    # Plot lines connecting the same 'bank' points for each subject
    for subj in df['subj'].unique():
        subj_data = df[df['subj'] == subj]
        for m in range(len(band_choice_1[n])):
            band_data = subj_data[subj_data['band'] == band_choice_1[n][m]]
            c1_data = band_data[band_data['pe_type']==pes[0]]
            c2_data = band_data[band_data['pe_type']==pes[1]]
            if len(band_data) > 1:
                axes[n].plot([m-0.2,m+0.2], \
                [c1_data['power'],c2_data['power']], color='grey', \
                    alpha=.2, linestyle='--', linewidth=1)
    
    df_ = df[df['band']==band_choice_1[n][m]]
    print(pes[0], np.asarray(cal_mean_sem(df_[df_['pe_type']==pes[0]]['power'])), pes[1],\
                   np.asarray(cal_mean_sem(df_[df_['pe_type']==pes[1]]['power'])))            
    # error bar
    sns.pointplot(ax=axes[n],\
                    data=df, x='band', y='power', order=band_choice_1[n],\
                    hue='pe_type', hue_order= pes, \
                    dodge=0.4, palette=[colors[pes[0]], colors[pes[1]]], \
                    linestyles='none',capsize=0.1, \
                    errorbar='se', err_kws={'linewidth': 0.9})
    for line in axes[n].lines:
        line.set_markersize(1)
        
    n_ch = np.where(np.array(ch_list)==chs_1[n])[0][0]
    anots = []
    pairs = []
    for f in band_choice_1[n]:
        anots.append(convert_pvalue_to_asterisks(p_value_pairs[n][n_ch][f]))
        print(p_value_pairs[n][n_ch][f])
        pairs.append(((f, pes[0]),(f, pes[1])))
        # ax[c].text(f, -88, convert_pvalue_to_asterisks(p_across_chns_1_sr[n_ch][f]),fontsize=16)

    axes[n].set_ylim(axes[n].get_ylim()[0], axes[n].get_ylim()[1])
    annot = Annotator(ax=axes[n], pairs=pairs, data=df, x='band',y='power',\
                      order=band_choice_1[n],\
                      hue='pe_type',hue_order = pes)
    annot.configure(loc='inside', verbose=0, fontsize=12,text_offset=0, line_width=.8)
    annot.set_custom_annotations(anots)
    annot.annotate()


    # axes[n].set_ylim(axes[n].get_ylim()[0], axes[n].get_ylim()[1]+2)

    axes[n].set_xlim([-.5,m+.5])
    axes[n].set_title(f'{tenten_chs_1[n]}', fontsize=8, y=.95)
    axes[n].set_yticks(axes[n].get_yticks(), axes[n].get_yticks().astype(int), fontsize=8)
    axes[n].set_xticks(range(len(band_choice_1[n])), [band_list_simple[j] for j in band_choice_1[n]], fontsize=8)

    # Adjust ticklabel padding
    axes[n].tick_params(axis='x', pad=2)  # Decrease the x-axis padding
    axes[n].tick_params(axis='y', pad=2)  # Decrease the y-axis padding
    
    axes[n].set_xlabel('',fontsize=8)
    if n == 0:
        axes[n].set_ylabel('Power [dB]',fontsize=8)
    else:
        axes[n].set_ylabel('')

    # axes[n].legend(loc=(.1,.1))
    # handles, labels = axes[n].get_legend_handles_labels()  # Get current legend handles and labels
    # axes[n].legend(handles[2:], labels[2:], title='PE type', title_fontsize=14, fontsize=12)
    axes[n].legend().remove()

plt.suptitle('Binary choice task', y=1,fontsize=8)
plt.tight_layout(pad=0.15,w_pad=.2,h_pad=.2)
# plt.subplots_adjust(left=0.1, right=0.9, top=0.8, bottom=0.2)
plt.gcf().set_dpi(300)
plt.savefig('../../results/fig4abBinary')
plt.show()

In [ ]:
# [FINAL-final] horizontal - 1 against 1: picked
# navigation task
chs_2 = ['E9','E6','E16','E57']
tenten_chs_2 = ['F1', 'Fz','C1','FC6']
band_choice_2 = [[1],[1],[1],[0]]

pe_pairs = [['SPE','nonPE'],['RPE','nonPE'],['dualPE', 'nonPE'],['SPE','RPE']]
com_list = ['s10','r10','sr10','s1vr1']
p_value_pairs = [p_across_chns_2_SPE_nonPE, p_across_chns_2_RPE_nonPE, \
                 p_across_chns_2_dualPE_nonPE, p_across_chns_2_SPE_RPE]

this_df = all_psd_2_df
group_df = this_df.groupby(['pe_type','band','ch','com'])['power'].mean().reset_index()
colors = {'SPE': cmap1[0], 'RPE': cmap1[1], 'nonPE': cmap1[2], 'dualPE': cmap1[3]}

cm = 1/2.54 
fig = plt.figure(figsize=(7*cm, 3.5*cm))
# Create a GridSPEc object with 2 rows and 2 columns, setting width and height ratios
gs = gridspec.GridSpec(1, 4)
# Create the first subplot, which spans the entire first row (bigger subplot)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])
ax4 = fig.add_subplot(gs[3])
axes = [ax1, ax2, ax3, ax4]


for n in range(len(pe_pairs)):
    df = this_df[(this_df['ch']==chs_2[n])&(this_df['com']==com_list[n])]
    pes = pe_pairs[n]
    # scatter for each subject
    x_pe = -1
    for pe in [pes[1],pes[0]]:
        for b in range(len(band_choice_2[n])):
            pe_data = df[(df['pe_type'] == pe)&(df['band']==band_choice_2[n][b])]
            axes[n].scatter([b+0.2*x_pe]*len(pe_data), pe_data['power'], label=pe,\
                            edgecolors='None', alpha=.2, color='grey', s=10)
        x_pe = 1
        
    
    # Manually adding lines between the error bars for the same x
    for i in range(len(band_choice_2[n])):
        pos1 = i + 0.2*(-1)
        pos2 = i + 0.2*(1)
        mean1 = df[(df['band'] == band_choice_2[n][i]) & (df['pe_type'] == pes[1])]['power'].mean()
        mean2 = df[(df['band'] == band_choice_2[n][i]) & (df['pe_type'] == pes[0])]['power'].mean()
        axes[n].plot([pos1, pos2], [mean1, mean2], color='black', linestyle='-',linewidth=.8)

    # Plot lines connecting the same 'bank' points for each subject
    for subj in df['subj'].unique():
        subj_data = df[df['subj'] == subj]
        for m in range(len(band_choice_2[n])):
            band_data = subj_data[subj_data['band'] == band_choice_2[n][m]]
            c1_data = band_data[band_data['pe_type']==pes[1]]
            c2_data = band_data[band_data['pe_type']==pes[0]]
            if len(band_data) > 1:
                axes[n].plot([m-0.2,m+0.2], \
                [c1_data['power'],c2_data['power']], color='grey', \
                    alpha=.2, linestyle='--', linewidth=1)
    
    df_ = df[df['band']==band_choice_2[n][m]]
    print(pes[0], np.asarray(cal_mean_sem(df_[df_['pe_type']==pes[0]]['power'])), pes[1],\
                   np.asarray(cal_mean_sem(df_[df_['pe_type']==pes[1]]['power'])))            
    # error bar
    sns.pointplot(ax=axes[n],\
                    data=df, x='band', y='power', order=band_choice_2[n],\
                    hue='pe_type', hue_order= [pes[1], pes[0]], \
                    dodge=0.4, palette=[colors[pes[1]], colors[pes[0]]], \
                    linestyles='none',capsize=0.1, \
                    errorbar='se', err_kws={'linewidth': 0.9})
    for line in axes[n].lines:
        line.set_markersize(1)            

    n_ch = np.where(np.array(ch_list)==chs_2[n])[0][0]
    anots = []
    pairs = []
    for f in band_choice_2[n]:
        anots.append(convert_pvalue_to_asterisks(p_value_pairs[n][n_ch][f]))
        print(p_value_pairs[n][n_ch][f])
        pairs.append(((f, pes[1]),(f, pes[0])))
        # ax[c].text(f, -88, convert_pvalue_to_asterisks(p_across_chns_1_sr[n_ch][f]),fontsize=16)

    axes[n].set_ylim(axes[n].get_ylim()[0], axes[n].get_ylim()[1])
    annot = Annotator(ax=axes[n], pairs=pairs, data=df, x='band',y='power',\
                      order=band_choice_2[n],\
                      hue='pe_type',hue_order = [pes[1], pes[0]])
    annot.configure(loc='inside', fontsize=12,text_offset=0, line_width=.8, verbose=0)
    annot.set_custom_annotations(anots)
    annot.annotate()

    

    axes[n].set_xlim([-.5,m+.5])
    axes[n].set_title(f'{tenten_chs_2[n]}', fontsize=8, y=.95)
    axes[n].set_yticks(axes[n].get_yticks(), axes[n].get_yticks().astype(int), fontsize=8)
    axes[n].set_xticks(range(len(band_choice_2[n])), [band_list_simple[j] for j in band_choice_2[n]], fontsize=8)
    
    axes[n].set_xlabel('',fontsize=8)
    if n == 0:
        axes[n].set_ylabel('Power [dB]',fontsize=8)
    else:
        axes[n].set_ylabel('')

    # Adjust ticklabel padding
    axes[n].tick_params(axis='x', pad=2)  # Decrease the x-axis padding
    axes[n].tick_params(axis='y', pad=2)  # Decrease the y-axis padding

    # axes[n].legend(loc=(.1,.1))
    # handles, labels = axes[n].get_legend_handles_labels()  # Get current legend handles and labels
    # axes[n].legend(handles[2:], labels[2:], title='PE type', title_fontsize=14, fontsize=12)
    axes[n].legend().remove()


plt.suptitle('Navigation task', y=1,fontsize=8)
plt.tight_layout(pad=0.15,w_pad=.2,h_pad=.2)
# plt.subplots_adjust(left=0.1, right=0.9, top=0.8, bottom=0.2)
plt.gcf().set_dpi(300)
plt.savefig('../../results/fig4abNavigation')
plt.show()
# if you cannot see the figure from this notebook, 
# please check the figure from results/fig4abNavigation.svg

In [ ]:
# [FINAL-final] horizontal - 1 against rest: picked
chs = ['E19']
tenten_chs = ['FT7']
pe_order = ['RPE','dualPE','SPE']
band_choice = [3]
p_sig = p_across_chns_2_dualPE
p_value = np.squeeze(p_across_chns_2_dualPE_value,1)

this_df = all_psd_2_df
this_df = this_df[this_df['com']=='dualPEvsr']
group_df = this_df.groupby(['pe_type','band','ch'])['power'].mean().reset_index()
colors = {'SPE': cmap1[0], 'RPE': cmap1[1], \
          'nonPE': cmap1[2], 'dualPE': cmap1[3]}

cm = 1/2.54 
fig = plt.figure(figsize=(3*cm, 3.5*cm))
# Create a figure
cm = 1/2.54 
# Create a GridSPEc object with 1 rows and 2 columns, setting width and height ratios
gs = gridspec.GridSpec(1, 1)
# Create the first subplot, which spans the entire first row (bigger subplot)
ax1 = fig.add_subplot(gs[0])
ax = [ax1]

x = 'band'
y = 'power'
width = 0.3
palette=[colors[pe_order[0]], colors[pe_order[1]],\
                colors[pe_order[2]]]
hue_order = pe_order

for ch in range(len(chs)):
    band = band_choice[ch]
    df = this_df[(this_df['ch']==chs[ch])&(this_df['band']==band)]


    n_ch = np.where(np.array(ch_list)==chs[ch])[0][0]
    anots = []
    pairs = []
    a = 0
    for m in range(2):
        anots.append(convert_pvalue_to_asterisks(\
            p_value[n_ch][band][m]))
        print(p_value[n_ch][band][m])
        pairs.append(((band, pe_order[1]),(band, pe_order[a])))
        a = 2

    
    ax[ch].set_ylim(ax[ch].get_ylim()[0]-1.3, ax[ch].get_ylim()[1]+2)


    # scatter for each subject
    power_mean = []
    x_pe = -.25
    for pe in pe_order:
        pe_data = df[(df['pe_type'] == pe)]
        ax[ch].scatter([x_pe]*len(pe_data), pe_data['power'], label=pe,\
                        edgecolors='None', alpha=.2, color='grey', s=10).set_zorder(0)
        power_mean.append(pe_data['power'].mean())
        x_pe += .25


    # Manually adding lines between the error bars for the same x
    ax[ch].plot([-0.25, 0, 0.25], power_mean, color='black', linestyle='-', linewidth=.8)

    # Plot lines connecting the same 'bank' points for each subject
    for subj in df['subj'].unique():
        band_data = df[df['subj'] == subj]
        c1_data = band_data[band_data['pe_type']==pe_order[0]]
        c2_data = band_data[band_data['pe_type']==pe_order[1]]
        c3_data = band_data[band_data['pe_type']==pe_order[2]]
        if len(band_data) > 1:
            ax[ch].plot([-0.25,0,0.25], \
            [c1_data['power'].iloc[0],c2_data['power'].iloc[0],c3_data['power'].iloc[0]], color='grey', \
                alpha=.2, linestyle='--', linewidth=1)

    df_ = df
    print(pe_order[0], np.asarray(cal_mean_sem(df_[df_['pe_type']==pe_order[0]]['power'])),\
           pe_order[1], np.asarray(cal_mean_sem(df_[df_['pe_type']==pe_order[1]]['power'])),\
           pe_order[2], np.asarray(cal_mean_sem(df_[df_['pe_type']==pe_order[2]]['power'])))
              
    # error bar
    sns.pointplot(ax=ax[ch],data=df, x='band',y='power', \
                  hue='pe_type', hue_order=hue_order, palette=palette,\
                    dodge=.5,linestyles='none',capsize=0.1, \
                    errorbar='se', err_kws={'linewidth': 0.9})
    for line in ax[ch].lines:
        line.set_markersize(1)
    
    annot = Annotator(ax=ax[ch],pairs=pairs,\
                    data=df, x='band', y='power', \
                    hue='pe_type', hue_order= hue_order)
    annot.configure(loc='inside', verbose=0, fontsize=12,text_offset=0, line_width=.8)
    annot.set_custom_annotations(anots)
    annot.annotate()


    ax[ch].set_xlim([-.5,.5])
    ax[ch].set_title(f'{tenten_chs[ch]}', fontsize=8, y=.95)

    ax[ch].set_yticks(ax[ch].get_yticks(), ax[ch].get_yticks().astype(int), fontsize=8)
    ax[ch].set_xticks(ax[ch].get_xticks(), [band_list_simple[band]],fontsize=8)
    # Adjust ticklabel padding
    ax[ch].tick_params(axis='x', pad=2)  # Decrease the x-axis padding
    ax[ch].tick_params(axis='y', pad=2)  # Decrease the y-axis padding
    ax[ch].set_xlabel('',fontsize=8)

    if ch==0:
        ax[ch].set_ylabel('Power [dB]',fontsize=8) #Power [dB]
    else:
        ax[ch].set_ylabel('') #Power [dB]
 
# handles, labels = ax.get_legend_handles_labels()  # Get current legend handles and labels
# ax.legend(handles[:4], labels[:4], title='PE type', \
#                title_fontsize=20, fontsize=20, loc=(1.1,1))
    ax[ch].legend().remove()



# plt.suptitle('2D', y=1,fontsize=8)
plt.tight_layout(pad=0.15,w_pad=.2,h_pad=.2)
# plt.subplots_adjust(left=0.1, right=0.9, top=0.8, bottom=0.2)
plt.gcf().set_dpi(300)
plt.savefig('../../results/fig4c.png')
plt.show()

# if you cannot see the figure from this notebook, 
# please check the figure from results/fig4c.svg